# 📖 Bible Reading Progress Tracker - WhatsApp Parser Validation

**Purpose:**  

Validate the `WhatsAppParser` against known input and edge cases before running the full pipeline


| # | Section | Description |
|---|---------|-------------|
| 1 | **Setup** | Imports and synthetic test data |
| 2 | **Platform Detection** | iOS vs Android format recognition |
| 3 | **Message Extraction** | Basic parsing, multiline, system messages |
| 4 | **Timestamp Parsing** | Format correctness, invalid dates, coercion |
| 5 | **Edge Cases** | Emoji, special characters, colons in messages |
| 6 | **Real File Smoke Test** | Run against actual export file |
| 7 | **Summary** | Pass/fail report |

---
## 0. Setup

In [1]:
import sys
import warnings
import pandas as pd
from pathlib import Path
from io import StringIO

warnings.filterwarnings('ignore')
sys.path.append(str(Path('..').resolve() / 'src'))

from preprocessing import WhatsAppParser

parser = WhatsAppParser()

# ── Test result tracker ──────────────────────────────────────────────────────
results = []  # list of (section, test_name, passed, note)

def check(section: str, name: str, condition: bool, note: str = ''):
    """Record a test result and print immediate feedback."""
    status = '✅ PASS' if condition else '❌ FAIL'
    results.append((section, name, condition, note))
    print(f'{status}  [{section}] {name}' + (f' — {note}' if note else ''))

print('Setup Complete.')

Setup Complete.


### Synthetic Test Data

In [2]:
# ── iOS samples ──────────────────────────────────────────────────────────────
IOS_BASIC = """[01/08/22 07.30.00] ~ Jason: Ef 1-2 done
[01/08/22 07.31.00] ~ Budi: Gal 3-6 selesai
"""

IOS_MULTILINE = """[01/08/22 08.00.00] ~ Jason: Ef 1-2 done
Lanjut besok ya
GBU all
[01/08/22 08.01.00] ~ Budi: Gal 3-6 selesai
"""

IOS_SYSTEM = """[01/08/22 07.00.00] ~ Jason: Ef 1-2 done
[01/08/22 07.01.00] ~ Jason: Jason added Budi
[01/08/22 07.02.00] ~ Budi: Gal 3-6 selesai
"""

# ── Android samples ──────────────────────────────────────────────────────────
ANDROID_BASIC = """01/08/22, 07:30 - Jason: Ef 1-2 done
01/08/22, 07:31 - Budi: Gal 3-6 selesai
"""

ANDROID_MULTILINE = """01/08/22, 08:00 - Jason: Ef 1-2 done
Lanjut besok ya
GBU all
01/08/22, 08:01 - Budi: Gal 3-6 selesai
"""

ANDROID_SYSTEM = """01/08/22, 07:00 - Jason: Ef 1-2 done
01/08/22, 07:01 - Messages and calls are end-to-end encrypted.
01/08/22, 07:02 - Budi: Gal 3-6 selesai
"""

# ── Edge cases ───────────────────────────────────────────────────────────────
EDGE_EMOJI = """01/08/22, 07:30 - Jason: Ef 1-2 done 🙏✅
01/08/22, 07:31 - Budi: 2 Kor 4-9🙏
"""

EDGE_COLON_IN_MESSAGE = """01/08/22, 07:30 - Jason: Note: Ef 1-2 done
01/08/22, 07:31 - Budi: Yoh 3:16 ingat ya
"""

EDGE_MARKDOWN = """01/08/22, 07:30 - Jason: _Ef 1-2_ done
01/08/22, 07:31 - Budi: *Gal 3-6* selesai
"""

EDGE_SENDER_WITH_TILDE = """[01/08/22 07.30.00] ~Jason Tan: Ef 1-2 done
[01/08/22 07.31.00] ~ Budi Santoso: Gal 3-6 selesai
"""

print('Test data defined.')

Test data defined.


---
## 1. Platform Detection

**Expected behavior**  
- Content starting with `[` → `'iOS'`  
- Content starting with `dd/mm/yyyy,` → `'Android'`

In [3]:
# iOS detection
platform = parser.detect_platform(IOS_BASIC)
check('Platform', 'iOS detected', platform == 'iOS', f'got: {platform}')

# Android detection
platform = parser.detect_platform(ANDROID_BASIC)
check('Platform', 'Android detected', platform == 'Android', f'got: {platform}')

# Empty content defaults to Android
platform = parser.detect_platform('')
check('Platform', 'Empty defaults to Android', platform == 'Android', f'got: {platform}')

✅ PASS  [Platform] iOS detected — got: iOS
✅ PASS  [Platform] Android detected — got: Android
✅ PASS  [Platform] Empty defaults to Android — got: Android


## 2, Message Extraction

### 2.1 Basic Parsing

In [4]:
def parse_string(content: str) -> pd.DataFrame:
    """Helper to write synthetic content to a temp file and parse it."""
    tmp = Path('./tmp/test_chat.txt')
    tmp.parent.mkdir(parents=True, exist_ok=True)
    tmp.write_text(content, encoding='utf-8')
    return parser.parse_chat_file(tmp)

# iOS Basic
df = parse_string(IOS_BASIC)
check('Extraction', 'iOS — correct row count', len(df) == 2, f'got: {len(df)}')
check('Extraction', 'iOS — columns present', set(['timestamp', 'sender', 'message']).issubset(df.columns))
check('Extraction', 'iOS — sender parsed', df.iloc[0]['sender'] == 'Jason',  f'got: {df.iloc[0]["sender"]}')
check('Extraction', 'iOS — message parsed', df.iloc[0]['message'] == 'Ef 1-2 done', f'got: {df.iloc[0]["message"]}')

# Android Basic
df = parse_string(ANDROID_BASIC)
check('Extraction', 'Android — correct row count', len(df) == 2, f'got: {len(df)}')
check('Extraction', 'Android — columns present', set(['timestamp', 'sender', 'message']).issubset(df.columns))
check('Extraction', 'Android — sender parsed', df.iloc[0]['sender'] == 'Jason', f'got: {df.iloc[0]["sender"]}')
check('Extraction', 'Android — message parsed', df.iloc[0]['message'] == 'Ef 1-2 done', f'got: {df.iloc[0]["message"]}')

✅ PASS  [Extraction] iOS — correct row count — got: 2
✅ PASS  [Extraction] iOS — columns present
✅ PASS  [Extraction] iOS — sender parsed — got: Jason
✅ PASS  [Extraction] iOS — message parsed — got: Ef 1-2 done
✅ PASS  [Extraction] Android — correct row count — got: 2
✅ PASS  [Extraction] Android — columns present
✅ PASS  [Extraction] Android — sender parsed — got: Jason
✅ PASS  [Extraction] Android — message parsed — got: Ef 1-2 done


In [5]:
df

,timestamp,sender,message
0,2022-08-01 07:30:00,Jason,Ef 1-2 done
1,2022-08-01 07:31:00,Budi,Gal 3-6 selesai


### 2.2 Multiline messages

In [6]:
# iOS multiline
df = parse_string(IOS_MULTILINE)
check('Multiline', 'iOS — row count unchanged', len(df) == 2, f'got: {len(df)}')
check('Multiline', 'iOS — continuation appended', '\n' in df.iloc[0]['message'],
      f'got: {repr(df.iloc[0]["message"])}')
check('Multiline', 'iOS — full content preserved', 'GBU all' in df.iloc[0]['message'],
      f'got: {repr(df.iloc[0]["message"])}')

# Android multiline
df = parse_string(ANDROID_MULTILINE)
check('Multiline', 'Android — row count unchanged', len(df) == 2, f'got: {len(df)}')
check('Multiline', 'Android — continuation appended', '\n' in df.iloc[0]['message'],
      f'got: {repr(df.iloc[0]["message"])}')

✅ PASS  [Multiline] iOS — row count unchanged — got: 2
✅ PASS  [Multiline] iOS — continuation appended — got: 'Ef 1-2 done\nLanjut besok ya\nGBU all'
✅ PASS  [Multiline] iOS — full content preserved — got: 'Ef 1-2 done\nLanjut besok ya\nGBU all'
✅ PASS  [Multiline] Android — row count unchanged — got: 2
✅ PASS  [Multiline] Android — continuation appended — got: 'Ef 1-2 done\nLanjut besok ya\nGBU all'


In [7]:
print('Multiline message content:')
print(repr(df.iloc[0]['message']))

Multiline message content:
'Ef 1-2 done\nLanjut besok ya\nGBU all'


### 2.3 System Messages

In [8]:
# iOS system messages
df = parse_string(IOS_SYSTEM)
print('iOS with system message:')
print(df.to_string())
print()

user_rows = df[df['sender'].notna()]
check('System', 'iOS — user messages present', len(user_rows) >= 2, f'got: {len(user_rows)}')


# Android system messages
df = parse_string(ANDROID_SYSTEM)
print('Android with system message:')
print(df.to_string())

user_rows = df[df['sender'].notna()]
check('System', 'Android — user messages present', len(user_rows) >= 2, f'got: {len(user_rows)}')

iOS with system message:
            timestamp sender           message
0 2022-08-01 07:00:00  Jason       Ef 1-2 done
1 2022-08-01 07:01:00  Jason  Jason added Budi
2 2022-08-01 07:02:00   Budi   Gal 3-6 selesai

✅ PASS  [System] iOS — user messages present — got: 3
Android with system message:
            timestamp sender                                       message
0 2022-08-01 07:00:00  Jason                                   Ef 1-2 done
1 2022-08-01 07:01:00   None  Messages and calls are end-to-end encrypted.
2 2022-08-01 07:02:00   Budi                               Gal 3-6 selesai
✅ PASS  [System] Android — user messages present — got: 2


---
## 3. Timestamp Parsing

**Expected behavior**  
- Timestamps parsed to `datetime64` without NaT on valid inputs  
- Invalid timestamps coerced to `NaT` without raising exceptions

In [9]:
# iOS timestamps
df = parse_string(IOS_BASIC)
check('Timestamp', 'iOS — dtype is datetime',   pd.api.types.is_datetime64_any_dtype(df['timestamp']),
      f'got: {df["timestamp"].dtype}')
check('Timestamp', 'iOS — no NaT in valid data', df['timestamp'].notna().all(),
      f'NaT count: {df["timestamp"].isna().sum()}')
check('Timestamp', 'iOS — correct date parsed',
      df.iloc[0]['timestamp'].date().isoformat() == '2022-08-01',
      f'got: {df.iloc[0]["timestamp"]}')
check('Timestamp', 'iOS — correct time parsed',
      df.iloc[0]['timestamp'].strftime('%H:%M:%S') == '07:30:00',
      f'got: {df.iloc[0]["timestamp"]}')

# Android timestamps
df = parse_string(ANDROID_BASIC)
check('Timestamp', 'Android — dtype is datetime', pd.api.types.is_datetime64_any_dtype(df['timestamp']),
      f'got: {df["timestamp"].dtype}')
check('Timestamp', 'Android — no NaT in valid data', df['timestamp'].notna().all(),
      f'NaT count: {df["timestamp"].isna().sum()}')
check('Timestamp', 'Android — correct date parsed',
      df.iloc[0]['timestamp'].date().isoformat() == '2022-08-01',
      f'got: {df.iloc[0]["timestamp"]}')

✅ PASS  [Timestamp] iOS — dtype is datetime — got: datetime64[ns]
✅ PASS  [Timestamp] iOS — no NaT in valid data — NaT count: 0
✅ PASS  [Timestamp] iOS — correct date parsed — got: 2022-08-01 07:30:00
✅ PASS  [Timestamp] iOS — correct time parsed — got: 2022-08-01 07:30:00
✅ PASS  [Timestamp] Android — dtype is datetime — got: datetime64[ns]
✅ PASS  [Timestamp] Android — no NaT in valid data — NaT count: 0
✅ PASS  [Timestamp] Android — correct date parsed — got: 2022-08-01 07:30:00


In [10]:
# Invalid timestamp coercion
INVALID_TIMESTAMP = """99/99/99, 99:99 - Jason: bad timestamp test
01/08/22, 07:30 - Budi: valid message
"""

try:
    df = parse_string(INVALID_TIMESTAMP)
    nat_count = df['timestamp'].isna().sum()
    check('Timestamp', 'Invalid date coerced to NaT', nat_count >= 1, f'NaT count: {nat_count}')
    check('Timestamp', 'Valid row still parsed', df['timestamp'].notna().sum() >= 1)
except Exception as e:
    check('Timestamp', 'Invalid date does not raise', False, str(e))

✅ PASS  [Timestamp] Invalid date coerced to NaT — NaT count: 1
✅ PASS  [Timestamp] Valid row still parsed


In [11]:
df

,timestamp,sender,message
0,NaT,Jason,bad timestamp test
1,2022-08-01 07:30:00,Budi,valid message


---
## 4. Edge Cases

### 4.1 Emoji in Messages

In [12]:
df = parse_string(EDGE_EMOJI)
check('Edge', 'Emoji — correct row count', len(df) == 2, f'got: {len(df)}')
check('Edge', 'Emoji — preserved in message', '🙏' in df.loc[0]['message'],
      f'got: {repr(df.loc[0]['message'])}')
check('Edge', 'Emoji — attached to chapter', '4-9🙏' in df.loc[1]['message'],
      f'got: {repr(df.loc[1]['message'])}')

✅ PASS  [Edge] Emoji — correct row count — got: 2
✅ PASS  [Edge] Emoji — preserved in message — got: 'Ef 1-2 done 🙏✅'
✅ PASS  [Edge] Emoji — attached to chapter — got: '2 Kor 4-9🙏'


### 4.2 Colon in Message Body

In [13]:
df = parse_string(EDGE_COLON_IN_MESSAGE)
check('Edge', 'Colon — correct row count', len(df) == 2, f'got: {len(df)}')
check('Edge', 'Colon — message body colon kept', 'Note:' in df.loc[0]['message'],
      f'got: {repr(df.loc[0]["message"])}')
check('Edge', 'Colon — verse ref kept', 'Yoh 3:16' in df.loc[1]['message'],
      f'got: {repr(df.loc[1]["message"])}')

✅ PASS  [Edge] Colon — correct row count — got: 2
✅ PASS  [Edge] Colon — message body colon kept — got: 'Note: Ef 1-2 done'
✅ PASS  [Edge] Colon — verse ref kept — got: 'Yoh 3:16 ingat ya'


### 4.3 Markdown Formatting

In [14]:
df = parse_string(EDGE_MARKDOWN)
check('Edge', 'Markdown — correct row count', len(df) == 2, f'got: {len(df)}')
check('Edge', 'Markdown — underscores preserved', '_Ef 1-2_' in df.loc[0]['message'],
      f'got: {repr(df.loc[0]["message"])}')
check('Edge', 'Markdown — asterisks preserved', '*Gal 3-6*' in df.loc[1]['message'],
      f'got: {repr(df.loc[1]["message"])}')

✅ PASS  [Edge] Markdown — correct row count — got: 2
✅ PASS  [Edge] Markdown — underscores preserved — got: '_Ef 1-2_ done'
✅ PASS  [Edge] Markdown — asterisks preserved — got: '*Gal 3-6* selesai'


### 4.4 Sender with Tilde Prefix

iOS exports sometimes include `~` or `~ ` before the sender name.

In [15]:
df = parse_string(EDGE_SENDER_WITH_TILDE)
check('Edge', 'Tilde — correct row count', len(df) == 2, f'got: {len(df)}')
check('Edge', 'Tilde — stripped from sender', '~' not in df.iloc[0]['sender'],
      f'got: {repr(df.iloc[0]["sender"])}')
check('Edge', 'Tilde — stripped with space', '~' not in df.iloc[1]['sender'],
      f'got: {repr(df.iloc[1]["sender"])}')

✅ PASS  [Edge] Tilde — correct row count — got: 2
✅ PASS  [Edge] Tilde — stripped from sender — got: 'Jason Tan'
✅ PASS  [Edge] Tilde — stripped with space — got: 'Budi Santoso'


### 4.5 Empty File

In [16]:
try:
    df = parse_string('')
    check('Edge', 'Empty File — returns empty DataFrame', df.empty)
    check('Edge', 'Empty File — correct columns',
           set(['timestamp', 'sender', 'message']).issubset(df.columns))
except Exception as e:
    check('Edge', 'Empty File — does not raise', False, str(e))

✅ PASS  [Edge] Empty File — returns empty DataFrame
✅ PASS  [Edge] Empty File — correct columns


---
## 5. Real File Smoke Test

Run the parser against the actual WhatsApp export.  
This is not a correctness test — just a sanity check that the real file parses without errors and produces reasonable output.

In [17]:
CHAT_FILE = Path('../data/raw/whatsapp_exports/_chat.txt')

if not CHAT_FILE.exists():
    check('Smoke', 'Chat file exists', False, f'Not found: {CHAT_FILE}')
else:
    try:
        df_real = parser.parse_chat_file(CHAT_FILE)

        check('Smoke', 'File parses without error', True)
        check('Smoke', 'Returns non-empty DataFrame', not df_real.empty,
              f'rows: {len(df_real)}')
        check('Smoke', 'No NaT timestamps',
              df_real['timestamp'].notna().mean() > 0.95,
              f'NaT rate: {df_real["timestamp"].isna().mean():.2%}')
        check('Smoke', 'No null senders > 5%',
              df_real['sender'].isna().mean() < 0.05,
              f'null sender rate: {df_real["sender"].isna().mean():.2%}')
        check('Smoke', 'No empty messages',
              (df_real['message'].str.strip() == '').mean() < 0.01,
              f'empty rate: {(df_real["message"].str.strip() == "").mean():.2%}')

        print(f'\nDataFrame shape  : {df_real.shape}')
        print(f'Date range         : {df_real["timestamp"].min()} → {df_real["timestamp"].max()}')
        print(f'Unique senders     : {df_real["sender"].nunique()}')
        print(f'\nSample rows:')
        display(df_real.sample(5, random_state=42))

    except Exception as e:
        check('Smoke', 'File parses without error', False, str(e))

✅ PASS  [Smoke] File parses without error
✅ PASS  [Smoke] Returns non-empty DataFrame — rows: 19142
✅ PASS  [Smoke] No NaT timestamps — NaT rate: 0.00%
✅ PASS  [Smoke] No null senders > 5% — null sender rate: 0.00%
✅ PASS  [Smoke] No empty messages — empty rate: 0.00%

DataFrame shape  : (19142, 3)
Date range         : 2020-08-02 11:41:31 → 2022-07-01 15:18:49
Unique senders     : 53

Sample rows:


,timestamp,sender,message
1199,2020-08-30 08:25:16,susianawati309,Kej 47-48 done
6880,2021-03-04 18:13:30,Mfitri,"1 Taw 29 - 2 Taw 1 done\nAnin,1 Taw 29 - 2 Taw..."
3826,2020-11-23 21:05:40,🪸Martha 🍁,Yos 2-7 done
1114,2020-08-27 21:37:53,Lenny Pandjidharma,Kej 43-44 done
3920,2020-11-27 08:41:27,Seto Ninik,Yos 14-15 done


---
## 6. Summary

In [18]:
summary = pd.DataFrame(results, columns=['Section', 'Test', 'Passed', 'Note'])

total = len(summary)
passed = summary['Passed'].sum()
failed = total - passed

print(f'Results: {passed}/{total} passed', '✅' if failed == 0 else f'— {failed} failed ❌')

if failed > 0:
    print('Failed tests:')
    print(summary[~summary['Passed']].to_string(index=False))

print('Full results by section:')
display(
    summary.style.applymap(
        lambda v: 'color: green' if v is True else ('color: red' if v is False else ''),
        subset=['Passed']
    )
)

Results: 46/46 passed ✅
Full results by section:


,Section,Test,Passed,Note
0,Platform,iOS detected,True,got: iOS
1,Platform,Android detected,True,got: Android
2,Platform,Empty defaults to Android,True,got: Android
3,Extraction,iOS — correct row count,True,got: 2
4,Extraction,iOS — columns present,True,
5,Extraction,iOS — sender parsed,True,got: Jason
6,Extraction,iOS — message parsed,True,got: Ef 1-2 done
7,Extraction,Android — correct row count,True,got: 2
8,Extraction,Android — columns present,True,
9,Extraction,Android — sender parsed,True,got: Jason
